# Massive Multilingual Speech (MMS) karaoke finetune

In [ ]:
import json
import re
import unicodedata
from typing import Any, cast

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from numpy.typing import NDArray
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    set_seed,
)
from transformers.modeling_outputs import CausalLMOutput

In [ ]:
MODEL_CHECKPOINT = "mms-300m"
MODEL_BASE = f"facebook/{MODEL_CHECKPOINT}"
MODEL_OUTPUT_DIR = f"{MODEL_CHECKPOINT}-ForcedAligner-karaoke-ja-Latn"

MAX_DURATION_SECONDS = 300
N_EVAL = 1000

BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
N_STEPS_PER_EPOCH = 20_000 // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
N_EPOCHS = 10
LEARNING_RATE = 5e-4

set_seed(42)

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset(
    "NextFire/karaoke-mugen-timings",
    split="train",
    streaming=True,
)

In [ ]:
def filter_by_duration(audio):
    return audio.metadata.duration_seconds <= MAX_DURATION_SECONDS


dataset = dataset.filter(filter_by_duration, input_columns=["audio"])

In [ ]:
vocab_dict = {v: k for k, v in enumerate("|abcdefghijklmnopqrstuvwxyz'")}
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    word_delimiter_token="|",
    unk_token="[UNK]",
    pad_token="[PAD]",
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(MODEL_OUTPUT_DIR)

In [ ]:
class Wav2Vec2ForForcedAligner(Wav2Vec2ForCTC):
    z_loss_weight: float = 1e-4

    def forward(
        self,
        input_values: torch.Tensor | None,
        attention_mask: torch.Tensor | None = None,
        output_attentions: bool | None = None,
        output_hidden_states: bool | None = None,
        return_dict: bool | None = None,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        outputs = super().forward(
            input_values=input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            labels=None,
            **kwargs,
        )
        assert isinstance(outputs, CausalLMOutput)
        assert outputs.logits is not None
        assert labels is not None
        ce = F.cross_entropy(
            outputs.logits.transpose(1, 2),
            labels,
            ignore_index=-100,
            reduction="none",
        )
        ce = ce[labels != -100]
        pt = torch.exp(-ce)
        focal_loss = ((1 - pt) ** 2 * ce).mean()
        log_z = torch.logsumexp(outputs.logits, dim=-1)
        z_loss = self.z_loss_weight * (log_z**2).mean()
        loss = focal_loss + z_loss
        return CausalLMOutput(
            loss=loss,  # pyright: ignore[reportArgumentType]
            logits=outputs.logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


model = Wav2Vec2ForForcedAligner.from_pretrained(
    MODEL_BASE,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    attn_implementation="kernels-community/flash-attn2",
)

# model.freeze_feature_encoder()

In [ ]:
def normalize_str(text: str):
    text = text.casefold()
    text = text.replace("’", "'")
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z']", " ", text)
    text = re.sub(r" +", " ", text)
    return text


def prepare_dataset(example: dict[str, Any]) -> dict[str, Any]:
    audio = example["audio"]
    inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],  # pyright: ignore[reportCallIssue]
    )
    input_values = cast(NDArray[np.float32], inputs.input_values[0])
    example["input_values"] = input_values

    output_length = int(model._get_feat_extract_output_lengths(len(input_values)))
    duration_ms = float(audio.metadata.duration_seconds * 1000)
    frames_per_ms = output_length / duration_ms
    pad_token_id = cast(int, processor.tokenizer.pad_token_id)
    labels = [pad_token_id] * output_length
    for line in example["timings"]:
        for timing in line:
            text = normalize_str(timing["text"])
            if not text:
                continue
            if len(text) >= 7 or text.strip().count(" ") > 0:
                # Suspicious alignment, skip it
                input_ids = [-200]
            else:
                input_ids = cast(list[int], processor(text=text).input_ids)
            start_frame = round(timing["start"] * frames_per_ms)
            assert start_frame < output_length
            end_frame = round(timing["end"] * frames_per_ms)
            assert start_frame <= end_frame <= output_length
            n_frames = end_frame - start_frame
            for idx, frame in enumerate(range(start_frame, end_frame)):
                token_idx = (idx * len(input_ids)) // n_frames
                token = input_ids[token_idx]
                if labels[frame] in (pad_token_id, -200):
                    labels[frame] = token
                elif token != -200:
                    # Mask overlapping alignments with -100 to ignore them in loss calculation
                    labels[frame] = -100
    labels = [label if label != -200 else -100 for label in labels]
    example["labels"] = labels

    return example


dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

In [ ]:
def data_collator(examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    input_features = [{"input_values": e["input_values"]} for e in examples]
    labels = [{"input_ids": e["labels"]} for e in examples]
    batch = processor.pad(input_features, padding=True, return_tensors="pt")
    l_batch = processor.pad(labels=labels, padding=True, return_tensors="pt")
    batch["labels"] = l_batch["input_ids"].masked_fill(
        l_batch.attention_mask.ne(1), -100
    )
    return batch


def compute_metrics(eval_pred: EvalPrediction):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=-1)
    valid_mask = labels != -100
    frame_accuracy = (predictions[valid_mask] == labels[valid_mask]).mean()
    return {"frame_accuracy": frame_accuracy}

In [ ]:
eval_dataset = dataset.take(N_EVAL)
train_dataset = dataset.skip(N_EVAL)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    max_steps=N_STEPS_PER_EPOCH * N_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=N_STEPS_PER_EPOCH // 4,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=N_STEPS_PER_EPOCH // 4,
    eval_on_start=True,
    save_strategy="steps",
    save_steps=N_STEPS_PER_EPOCH // 4,
    save_total_limit=5,
    metric_for_best_model="frame_accuracy",
    greater_is_better=True,
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="trackio",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,  # pyright: ignore[reportArgumentType]
    eval_dataset=eval_dataset,  # pyright: ignore[reportArgumentType]
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [ ]:
trainer.train()
trainer.save_model()